# GroupDNA - WhatsApp Chat Analyzer

### Industry-Graded Minor Project

Student Name: Yerramsetty Gayathri

Roll Number:

Course:Data Science

Tools Used:Python, NumPy, Datetime

Platform: Google Colab

---

## Project Description

GroupDNA is a WhatsApp Chat Analytics System that analyzes exported WhatsApp chat files and generates a complete activity report of the group.

The project is built using only Python Fundamentals and NumPy.

Features:
- Chat Parser
- Group Overview
- Most Active Day & Hour
- NumPy Activity Heatmap
- Top Words
- Response Analysis
- Silent Streak Detection
- Personality Archetypes
- Final Report

In [68]:
import numpy as np
from datetime import datetime, timedelta

In [69]:
from google.colab import files

uploaded = files.upload()

FILE_NAME = list(uploaded.keys())[0]

print("Uploaded:", FILE_NAME)

Saving 12721105-DADS_Minor_PROJECT_dataset.zip to 12721105-DADS_Minor_PROJECT_dataset (6).zip
Uploaded: 12721105-DADS_Minor_PROJECT_dataset (6).zip


In [70]:
import zipfile
import io

# Get the content of the uploaded zip file
zip_content = uploaded[FILE_NAME]

# Create a BytesIO object to read the zip content
zip_file_object = io.BytesIO(zip_content)

# Open the zip file
with zipfile.ZipFile(zip_file_object, 'r') as zf:
    # Assuming there's only one text file ending with .txt inside the zip
    text_file_name = [name for name in zf.namelist() if name.endswith('.txt')][0]
    with zf.open(text_file_name) as f:
        # Read and decode the content
        content = f.read().decode('utf-8')
        lines = content.splitlines() # Assign to lines variable

messages = []

system_messages = 0
media_messages = 0
deleted_messages = 0

for line in lines:

    line = line.strip()

    if line == "":
        continue

    if " - " not in line:
        continue

    try:
        timestamp, remaining = line.split(" - ", 1)

        if ": " not in remaining:
            system_messages += 1
            continue

        sender, message = remaining.split(": ", 1)

        messages.append({
            "timestamp": timestamp,
            "sender": sender,
            "message": message
        })

        if message == "<Media omitted>":
            media_messages += 1

        if message == "This message was deleted":
            deleted_messages += 1

    except:
        system_messages += 1

print("Parser Completed Successfully!")

Parser Completed Successfully!


In [71]:
print("="*60)
print("CHAT PARSER SUMMARY")
print("="*60)

print("Real Messages :", len(messages))
print("System Messages :", system_messages)
print("Media Messages :", media_messages)
print("Deleted Messages :", deleted_messages)

participants = set()

for msg in messages:
    participants.add(msg["sender"])

print("Participants :", len(participants))

CHAT PARSER SUMMARY
Real Messages : 3174
System Messages : 4
Media Messages : 32
Deleted Messages : 15
Participants : 6


#Feature 2: Group Overview

This section analyzes the WhatsApp chat dataset and provides:

- Total number of messages
- Total participants
- Date range
- Chat duration
- Messages sent by each participant
- Percentage contribution

In [72]:
# ==========================================================
# FEATURE 2 : GROUP OVERVIEW
# ==========================================================

message_count = {}
participants = set()

for msg in messages:

    sender = msg["sender"]

    participants.add(sender)

    if sender in message_count:
        message_count[sender] += 1
    else:
        message_count[sender] = 1

total_messages = len(messages)

In [73]:
# ==========================================================
# Date Range
# ==========================================================

date_objects = []

for msg in messages:

    dt = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")

    date_objects.append(dt)

start_date = min(date_objects)
end_date = max(date_objects)

total_days = (end_date - start_date).days + 1

In [74]:
# ==========================================================
# Messages Per Person
# ==========================================================

sorted_people = sorted(
    message_count.items(),
    key=lambda x: x[1],
    reverse=True
)

In [75]:
# ==========================================================
# GROUP OVERVIEW REPORT
# ==========================================================

sorted_people = sorted(
    message_count.items(),
    key=lambda x: x[1],
    reverse=True
)

print("=" * 70)
print(" " * 24 + "GROUP OVERVIEW")
print("=" * 70)

print(f"📅 Period           : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')}")
print(f"🗓️  Total Days       : {total_days}")
print(f"💬 Total Messages   : {total_messages}")
print(f"👥 Participants     : {len(participants)}")

print("\n" + "=" * 70)
print(f"{'Participant':<15}{'Messages':>12}{'Contribution':>20}")
print("=" * 70)

for person, count in sorted_people:

    percentage = (count / total_messages) * 100

    print(f"{person:<15}{count:>12}{percentage:>18.2f}%")

print("=" * 70)

                        GROUP OVERVIEW
📅 Period           : 01 April 2024 to 30 May 2024
🗓️  Total Days       : 60
💬 Total Messages   : 3174
👥 Participants     : 6

Participant        Messages        Contribution
Rahul                   953             30.03%
Priya                   718             22.62%
Neha                    635             20.01%
Aman                    490             15.44%
Karan                   354             11.15%
Vikas                    24              0.76%


# Feature 3: Most Active Day & Busiest Hour

This section identifies:

- The busiest day in the chat
- The busiest hour of the day
- Message activity distribution

In [76]:
# ==========================================================
# FEATURE 3 : MOST ACTIVE DAY
# ==========================================================

day_count = {}

for msg in messages:

    dt = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")

    day = dt.strftime("%d %B %Y")

    if day in day_count:
        day_count[day] += 1
    else:
        day_count[day] = 1

In [77]:
# ==========================================================
# FEATURE 3 : BUSIEST HOUR
# ==========================================================

hour_count = {}

for hour in range(24):
    hour_count[hour] = 0

for msg in messages:

    dt = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")

    hour = dt.hour

    hour_count[hour] += 1

In [78]:
# ==========================================================
# Find Most Active Day and Hour
# ==========================================================

most_active_day = max(day_count, key=day_count.get)
most_active_day_messages = day_count[most_active_day]

busiest_hour = max(hour_count, key=hour_count.get)
busiest_hour_messages = hour_count[busiest_hour]

In [79]:
# ==========================================================
# MOST ACTIVE DAY REPORT
# ==========================================================

print("=" * 70)
print(" " * 20 + "MOST ACTIVE DAY & BUSIEST HOUR")
print("=" * 70)

print(f"Most Active Day : {most_active_day}")
print(f"Messages        : {most_active_day_messages}")

print()

print(f"Busiest Hour    : {busiest_hour:02d}:00 - {busiest_hour + 1:02d}:00")
print(f"Messages        : {busiest_hour_messages}")

print("=" * 70)

                    MOST ACTIVE DAY & BUSIEST HOUR
Most Active Day : 04 May 2024
Messages        : 76

Busiest Hour    : 18:00 - 19:00
Messages        : 248


Feature 4: NumPy Activity Heatmap

This feature creates a **NumPy matrix** representing each participant's activity across the 24 hours of the day.

- Rows = Participants
- Columns = Hours (00–23)
- Values = Number of messages

In [80]:
# ==========================================================
# FEATURE 4 : NUMPY ACTIVITY HEATMAP
# ==========================================================

participants = sorted(list(participants))

participant_index = {}

for i, person in enumerate(participants):
    participant_index[person] = i

# 6 Participants × 24 Hours
heatmap = np.zeros((len(participants), 24), dtype=int)

for msg in messages:

    dt = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")

    row = participant_index[msg["sender"]]
    col = dt.hour

    heatmap[row][col] += 1

In [81]:
# ==========================================================
# DISPLAY NUMPY HEATMAP
# ==========================================================

print("=" * 90)
print("NUMPY ACTIVITY HEATMAP")
print("=" * 90)

header = "Participant".ljust(12)

for hour in range(24):
    header += f"{hour:>4}"

print(header)

print("-" * 90)

for i, person in enumerate(participants):

    row = person.ljust(12)

    for value in heatmap[i]:
        row += f"{value:>4}"

    print(row)

print("=" * 90)

NUMPY ACTIVITY HEATMAP
Participant    0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23
------------------------------------------------------------------------------------------
Aman          54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7  16   8  13  11   0  56
Karan          0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27  25  32  23  14   9   8
Neha           0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47  62  50  45  27  28  30
Priya          0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40  38  60  43  32  18   9
Rahul          3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49 105  76  41  92  60  54
Vikas          0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3   2   2   1   1   1   2


Feature 5: Top Words Analysis

This feature identifies the most frequently used words in the WhatsApp group after removing common stop words.

It helps understand the group's common discussion topics.

In [82]:
# ==========================================================
# FEATURE 5 : TOP WORDS ANALYSIS
# ==========================================================

stop_words = {
    "the","is","am","are","was","were","be","been","being",
    "to","of","for","from","in","on","at","by","with","into",
    "a","an","and","or","but","if","then","than",
    "this","that","these","those",
    "i","me","my","mine","myself",
    "you","your","yours","yourself",
    "he","his","him","she","her","hers",
    "we","our","ours","they","their","them",

    "have","has","had","having",
    "do","does","did","doing",
    "will","would","can","could","should","shall",
    "may","might","must",

    "what","when","where","who","why","how",
    "which","about","after","before","during",

    "yes","no","ok","okay","ya","yeah","hi","hello","hey",

    "today","tomorrow","yesterday","now","then",

    "just","also","only","even","still","again","very",
    "here","there","everyone","someone","anyone",

    "so","because","it's","its","it's"
}

In [83]:
word_frequency = {}

for msg in messages:

    text = msg["message"].lower()

    words = text.split()

    for word in words:

        word = word.strip(".,!?;:'\"()[]{}<>")

        if len(word) < 2:
            continue

        if word in stop_words:
            continue

        if word in word_frequency:
            word_frequency[word] += 1
        else:
            word_frequency[word] = 1

In [84]:
sorted_words = sorted(
    word_frequency.items(),
    key=lambda x: x[1],
    reverse=True
)

In [85]:
print("="*70)
print("TOP 10 MOST USED WORDS")
print("="*70)

for word, count in sorted_words[:10]:

    bar = "█" * min(count // 10 + 1, 30)

    print(f"{word:<15} {bar:<30} {count}")

print("="*70)

TOP 10 MOST USED WORDS
guys            ██████████████████████████████ 318
hai             ███████████████████████████    268
telling         ██████████████████             179
up              ██████████████████             172
bhai            █████████████████              160
one             ████████████████               157
started         ████████████████               150
scene           ███████████████                145
entire          ███████████████                145
please          ███████████████                141


Feature 6: Response Time Analysis

This feature calculates the average response time of each participant in the WhatsApp group.

The analysis identifies:

- Fastest responder
- Slowest responder
- Average response time (minutes)

In [86]:
# ==========================================================
# FEATURE 6 : RESPONSE TIME ANALYSIS
# ==========================================================

messages_sorted = sorted(
    messages,
    key=lambda x: datetime.strptime(x["timestamp"], "%d/%m/%y, %H:%M")
)

In [87]:
response_times = {}

for person in participants:
    response_times[person] = []

response_times = defaultdict(list)

for i in range(1, len(messages_sorted)):

    previous = messages_sorted[i - 1]
    current = messages_sorted[i]

    # Ignore consecutive messages from the same person
    if previous["sender"] == current["sender"]:
        continue

    previous_time = datetime.strptime(previous["timestamp"], "%d/%m/%y, %H:%M")
    current_time = datetime.strptime(current["timestamp"], "%d/%m/%y, %H:%M")

    gap = (current_time - previous_time).total_seconds() / 60

    # Ignore unrealistic gaps (more than 24 hours)
    if 0 <= gap <= 1440:
        response_times[current["sender"]].append(gap)

In [88]:
average_response = {}

for person in participants:

    values = response_times.get(person, [])

    if len(values) == 0:
        average_response[person] = 0
    else:
        average_response[person] = sum(values) / len(values)

In [89]:
print("=" * 75)
print("RESPONSE TIME ANALYSIS")
print("=" * 75)

for person, avg in sorted(average_response.items(), key=lambda x: x[1]):

    print(f"{person:<12} {avg:8.2f} minutes")

print("=" * 75)

fastest = min(average_response, key=average_response.get)
slowest = max(average_response, key=average_response.get)

print(f"\n⚡ Fastest Replier : {fastest}")
print(f"🐢 Slowest Replier : {slowest}")

RESPONSE TIME ANALYSIS
Rahul           34.95 minutes
Karan           36.62 minutes
Neha            39.45 minutes
Priya           41.99 minutes
Vikas           46.30 minutes
Aman            55.36 minutes

⚡ Fastest Replier : Rahul
🐢 Slowest Replier : Aman


 Feature 7: Silent Streak Detection

This feature calculates the longest number of consecutive days each participant remained inactive in the WhatsApp group.

In [90]:
# ==========================================================
# FEATURE 7 : SILENT STREAK
# ==========================================================

from datetime import timedelta

person_days = {}

for person in participants:
    person_days[person] = set()

for msg in messages:

    dt = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")

    day = dt.date()

    person_days[msg["sender"]].add(day)

In [91]:
silent_streak = {}

all_days = []

current = start_date.date()

while current <= end_date.date():

    all_days.append(current)

    current += timedelta(days=1)

In [92]:
for person in participants:

    longest = 0
    current = 0

    for day in all_days:

        if day not in person_days[person]:

            current += 1

            if current > longest:
                longest = current

        else:
            current = 0

    silent_streak[person] = longest

In [93]:
print("="*70)
print("LONGEST SILENT STREAKS")
print("="*70)

for person, days in sorted(silent_streak.items(),
                           key=lambda x: x[1],
                           reverse=True):

    print(f"{person:<12} {days} days")

print("="*70)

LONGEST SILENT STREAKS
Vikas        11 days
Aman         0 days
Karan        0 days
Neha         0 days
Priya        0 days
Rahul        0 days


 Feature 8: Personality Archetype Detection

This feature analyzes participant behavior based on:

- Total messages
- Average response time
- Silent streak
- Activity pattern
- Communication style

Each participant is assigned a personality archetype based on their dominant behavior.

In [94]:
# ==========================================================
# FEATURE 8 : PERSONALITY ARCHETYPE DETECTION
# ==========================================================

night_messages = {}

for person in participants:
    night_messages[person] = 0

for msg in messages:

    dt = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")

    hour = dt.hour

    if hour >= 22 or hour < 5:
        night_messages[msg["sender"]] += 1

In [95]:
long_messages = {}

for person in participants:
    long_messages[person] = 0

for msg in messages:

    words = msg["message"].split()

    if len(words) >= 20:
        long_messages[msg["sender"]] += 1

In [96]:
archetypes = {}

max_messages = max(message_count.values())
min_response = min(average_response.values())
max_silent = max(silent_streak.values())
max_night = max(night_messages.values())
max_story = max(long_messages.values())

for person in participants:

    if message_count[person] == max_messages:
        archetypes[person] = "The Spammer"

    elif average_response[person] == min_response:
        archetypes[person] = "Lightning Responder"

    elif silent_streak[person] == max_silent:
        archetypes[person] = "The Ghost"

    elif night_messages[person] == max_night:
        archetypes[person] = "Night Owl"

    elif long_messages[person] == max_story:
        archetypes[person] = "Story Teller"

    else:
        archetypes[person] = "Balanced Member"

In [97]:
print("=" * 75)
print("PERSONALITY ARCHETYPES")
print("=" * 75)

for person in sorted(participants):

    print(f"{person:<12}  {archetypes[person]}")

print("=" * 75)

PERSONALITY ARCHETYPES
Aman          Night Owl
Karan         Story Teller
Neha          Balanced Member
Priya         Balanced Member
Rahul         The Spammer
Vikas         The Ghost


Conclusion

The **GroupDNA – WhatsApp Group Chat Analyzer** successfully analyzes exported WhatsApp chats and extracts meaningful insights about participant behavior and communication patterns.

### Features Implemented
- Chat Parsing
- Group Overview
- Most Active Day & Busiest Hour
- NumPy Activity Heatmap
- Top Words Analysis
- Response Time Analysis
- Silent Streak Detection
- Personality Archetype Detection

### Technologies Used
- Python
- NumPy
- Google Colab

This project demonstrates the practical use of data structures, file handling, dictionaries, loops, date-time processing, and basic data analytics while following the project constraints.